# 11 · Hierarchical replication: when 300 trials still equal one replica

v0.9 adds the layer that prevents repeated lower-level observations from masquerading as independent scientific replication. The question is no longer only whether an effect exists on held-out trials, but **at what level it recurs independently**.

The central rule is: lower-level repetition improves precision, not higher-level sample size.

## The claim chooses the unit

For a model-training-seed claim, model seeds are independent units. For a subject claim, subjects are independent units. The hierarchy beneath the claim is averaged recursively before the claim-level units are combined.

```text
model seed
  → subject
    → session
      → trial
```

This notebook deliberately gives one seed many more trials than the others. A flat trial average would overweight it; the v0.9 estimator does not.

In [ ]:
from neuros_mechint.benchmarks import (
    HierarchicalReplicationPolicy, HierarchicalReplicationSpec,
    ReplicationAxis, ReplicationCoordinates, ReplicationObservation,
    analyze_hierarchical_replication,
)

spec = HierarchicalReplicationSpec(
    study_id='tutorial-seed-replication',
    family_id='relative-isi-shared-computation',
    claim_axis=ReplicationAxis.MODEL_SEED,
    primary_metric='causal_recovery_margin',
    hierarchy=(
        ReplicationAxis.MODEL_SEED,
        ReplicationAxis.SUBJECT,
        ReplicationAxis.TRIAL,
    ),
    expected_direction=1,
    seed=19,
    policy=HierarchicalReplicationPolicy(
        min_independent_units=3, bootstrap_samples=500,
        min_sign_agreement=0.75, min_absolute_effect=0.1,
    ),
)

In [ ]:
observations = []
# Seed 0 has 100 trials; seeds 1 and 2 have only 4 each.
for seed, effect, trials in [(0, 0.80, 100), (1, 0.70, 4), (2, 0.60, 4)]:
    for trial in range(trials):
        observations.append(ReplicationObservation(
            observation_id=f'{seed}:{trial}',
            family_id='relative-isi-shared-computation',
            coordinates=ReplicationCoordinates(
                model_seed=seed, subject_id=f'subject:{seed}',
                trial_id=f'seed:{seed}/trial:{trial}',
            ),
            metrics={'causal_recovery_margin': effect},
        ))

result = analyze_hierarchical_replication(spec, observations)
primary = result.primary_estimate
print('independent model seeds:', primary.independent_units)
print('unit-balanced estimate:', round(primary.estimate, 3))
print('confidence interval:', (round(primary.ci_low, 3), round(primary.ci_high, 3)))
print('replicated:', result.decision.replicated)
# The estimate is 0.70, not the trial-weighted value near 0.79.

## The pseudoreplication falsification

Now create 300 beautifully consistent trials from **one** model seed. The within-seed effect is extremely precise, but a model-seed replication claim is not estimable because there is only one independent seed.

In [ ]:
pseudo = [
    ReplicationObservation(
        observation_id=f'pseudo:{trial}',
        family_id='relative-isi-shared-computation',
        coordinates=ReplicationCoordinates(
            model_seed=0, subject_id='subject:0', trial_id=f'trial:{trial}',
        ),
        metrics={'causal_recovery_margin': 0.9},
    )
    for trial in range(300)
]
pseudo_result = analyze_hierarchical_replication(spec, pseudo)
print('estimable:', pseudo_result.decision.estimable)
print('replicated:', pseudo_result.decision.replicated)
print('reasons:', pseudo_result.decision.reasons)

## Dose response and manifold assumptions

v0.9 also lets an intervention claim become harder to fool with a lucky endpoint. A mapped substitution can be evaluated over 0%, 25%, 50%, 75%, and 100% dose while explicitly recording whether replacement values came from a zero baseline, empirical donor, nearest neighbor, conditional resampler, generator, or another declared manifold assumption.

A coherent dose response is supporting evidence, not proof. The strongest workflow is now:

```text
v0.6 held-out evidence
→ v0.7 estimable factorial difference
→ v0.8 held-out causal correspondence
→ v0.9 independent-seed / subject / session replication
→ dose response + alternative valid projector/manifold controls
```

The question to carry forward is not 'how many perturbations did we run?' It is **which scientifically independent worlds did the mechanism survive?**